# Aprendizado de Máquina — Lista prática 05

## Árvores de Regressão e *Ensembles*

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

A Lista Teórica 05 deduziu que a variância de um *ensemble* trava num piso
$\rho\,v(x)$, e que reduzir $\rho$ vale mais do que aumentar $B$. Esta lista
**mede as três quantidades** — $v$, $\rho$ e o piso — e verifica se a floresta
aleatória faz o que promete.

> **a floresta não é ``bagging com mais aleatoriedade''. Ela paga viés para
> comprar correlação, e dá para ver o preço e a compra na mesma tabela.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — uma árvore, e a profundidade

Comece pelo tijolo. Carregue o `superconductivity.csv`, fique com 3 000
observações de treino, e ajuste árvores de várias profundidades.

In [ ]:
_nome = "superconductivity.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

df = pd.read_csv(_caminho)
X_todos = df.drop(columns="critical_temp").values
y_todos = df["critical_temp"].values

X_tr, X_te, y_tr, y_te = skm.train_test_split(
    X_todos, y_todos, train_size=3000, test_size=4000, random_state=2026)

print("treino", X_tr.shape, " teste", X_te.shape)

In [ ]:
print("  profundidade   folhas   R2 treino   R2 teste")
for prof in (2, 5, 10, None):
    arvore = DecisionTreeRegressor(max_depth=prof, random_state=0).fit(X_tr, y_tr)   # (a)
    print(f"  {str(prof):>12s}   {arvore.get_n_leaves():6d}   "
          f"{arvore.score(X_tr, y_tr):9.4f}   {arvore.score(X_te, y_te):8.4f}")      # (b)

Deve imprimir:

```
  profundidade   folhas   R2 treino   R2 teste
             2        4      0.6810     0.6602
             5       32      0.7923     0.7304
            10      468      0.9335     0.7838
          None     2594      0.9922     0.7721
```

A curva em U da Aula 01, agora com a profundidade no papel do grau: o $R^2$ de
teste sobe até a profundidade 10 e cai depois. E a árvore sem limite chega a
$R^2$ de treino $0{,}9922$ com **2 594 folhas** para 3 000 observações — pouco
mais de uma observação por folha, ou seja, quase interpolação.

Guarde o número $0{,}7838$: é o melhor que uma árvore sozinha consegue aqui.

---
## Exercício 2 — medindo $v$, $\rho$ e o piso

Aqui está o exercício central da lista. A fórmula da Lista Teórica 05 diz

$$\operatorname{Var}(\bar g(x)) \;=\; \rho\,v(x) \;+\; \frac{1-\rho}{B}\,v(x),$$

e queremos as três quantidades **medidas**.

Há uma sutileza que decide o experimento: $\rho$ é a correlação entre duas
árvores **sobre a aleatoriedade que as gerou** — o que inclui o sorteio do
conjunto de treino. Não dá para estimá-la de uma única floresta. É preciso
ajustar $R$ florestas em $R$ conjuntos de treino **independentes**:

- $v(x)$: variância entre as $B$ árvores, dentro de cada floresta;
- $\operatorname{Var}(\bar g(x))$: variância da predição da floresta, entre as $R$ florestas;
- $\rho$: isolado da fórmula, $\rho = \dfrac{\operatorname{Var}(\bar g) - v/B}{v - v/B}$.

Usamos a população sintética da figura da nota ($n=150$, $p=8$, ruído $1{,}5$),
porque ali dá para sortear conjuntos de treino à vontade.

In [ ]:
def alvo(M):
    return np.sin(1.5 * M[:, 0]) + 0.8 * M[:, 1] * M[:, 2] + 0.5 * M[:, 0] ** 2


n, d, ruido = 150, 8, 1.5
B, R = 100, 40

rng = np.random.default_rng(2026)
X0 = rng.uniform(-2, 2, size=(400, d))       # pontos onde medimos

print("  max_features       v(x)    Var(gbar)      rho     piso rho*v   EQM teste")
for mf, rotulo in [(None, "d (bagging)"), (1 / 3, "d/3 (floresta)"), (0.15, "0,15 d")]:
    rng = np.random.default_rng(2026)
    gbar = np.zeros((R, len(X0)))
    vs, eqm = [], []

    for r_ in range(R):
        X = rng.uniform(-2, 2, size=(n, d))
        y = alvo(X) + rng.normal(0, ruido, size=n)
        floresta = RandomForestRegressor(n_estimators=B, max_features=mf,        # (a)
                                         random_state=r_, n_jobs=-1).fit(X, y)
        # predicao de CADA arvore, separadamente
        P = np.column_stack([arv.predict(X0) for arv in floresta.estimators_])   # (b)
        vs.append(P.var(axis=1, ddof=1).mean())        # variancia ENTRE arvores
        gbar[r_] = P.mean(axis=1)                    # (c) a predicao do ensemble
        eqm.append(np.mean((floresta.predict(X0) - alvo(X0)) ** 2))

    v = np.mean(vs)
    var_gbar = gbar.var(axis=0, ddof=1).mean()         # variancia ENTRE florestas
    rho = (var_gbar - v / B) / (v - v / B)           # (d) isole rho da formula

    print(f"  {rotulo:16s} {v:7.3f}   {var_gbar:8.3f}   {rho:8.4f}   {rho * v:8.3f}"
          f"   {np.mean(eqm):8.4f}")

Deve imprimir:

```
  max_features       v(x)    Var(gbar)      rho     piso rho*v   EQM teste
  d (bagging)        3.281      0.510     0.1469      0.482     1.3729
  d/3 (floresta)     3.800      0.314     0.0735      0.279     1.2901
  0,15 d             3.993      0.248     0.0526      0.210     1.3855
```

Esta tabela é a teoria inteira da floresta aleatória, em três linhas. Leia coluna
por coluna:

- **$v(x)$ sobe** conforme `max_features` cai: $3{,}28 \to 3{,}80 \to 3{,}99$.
  É o **preço**. Proibir a árvore de considerar as melhores covariáveis em parte
  dos nós a torna individualmente pior e mais instável.
- **$\rho$ cai pela metade** do *bagging* para a floresta: $0{,}147 \to 0{,}0735$,
  e continua caindo até $0{,}053$. É a **compra**, e é exatamente o mecanismo que
  a nota descreve.
- **$\operatorname{Var}(\bar g)$ cai 38%** ($0{,}510 \to 0{,}314$). A compra vale
  mais que o preço: a correlação caiu proporcionalmente mais do que $v$ subiu.
- **O piso $\rho v$ já é 88% da variância total** com $B=100$
  ($0{,}279$ de $0{,}314$). Ou seja: acrescentar árvores a esta floresta não tem
  para onde ir. Só reduzir $\rho$ tem.

E a última coluna mostra que o ganho não é infinito: de $p/3$ para $0{,}15p$, o
$\rho$ continua caindo mas o **EQM piora** ($1{,}2901 \to 1{,}3855$). A partir de
certo ponto o viés das árvores individuais domina, e é por isso que
`max_features` é um hiperparâmetro a escolher por validação cruzada, e não uma
quantidade a minimizar.

> **Sua vez.** Repita a linha da floresta com $B=10$ em vez de $B=100$. O $\rho$
> medido muda? E a $\operatorname{Var}(\bar g)$? Compare com o que a fórmula
> prevê.

---
## Exercício 3 — $B$ grande na floresta: inofensivo

Na mesma população, veja o que acontece quando $B$ cresce. Guarde também o
**erro de treino** — é ele que mostra o mecanismo.

In [ ]:
rng = np.random.default_rng(12)
X = rng.uniform(-2, 2, size=(n, d))
y = alvo(X) + rng.normal(0, ruido, size=n)
X_ex3_te = rng.uniform(-2, 2, size=(4000, d))
y_ex3_te = alvo(X_ex3_te) + rng.normal(0, ruido, size=4000)

print("floresta (max_features=1/3):")
for Bf in (1, 10, 100, 800):
    m = RandomForestRegressor(n_estimators=Bf, max_features=1/3, random_state=0).fit(X, y)
    print(f"   B={Bf:4d}: teste {np.mean((y_ex3_te - m.predict(X_ex3_te)) ** 2):.4f}"
          f"   treino {np.mean((y - m.predict(X)) ** 2):.4f}")

Deve imprimir:

```
floresta (max_features=1/3):
   B=   1: teste 8.0399   treino 3.1169
   B=  10: teste 3.7384   treino 0.7179
   B= 100: teste 3.3510   treino 0.4849
   B= 800: teste 3.3262   treino 0.4630
```

São os números da figura da nota, e a semente é a mesma **de propósito**: o
exercício existe para você reproduzi-los.

A coluna do **treino** explica o comportamento melhor que a do teste. Ela
estaciona em $0{,}46$ e **não desce mais**: cada árvore nova é ajustada a uma
reamostra independente, e a média de mais árvores independentes não persegue
ruído nenhum. É por isso que $B$ grande aqui é inofensivo — ele só mexe na
variância, nunca no viés.

Repare também no $B=1$: uma árvore só dá $8{,}04$, quase o dobro de tudo o mais. O
*ensemble* não está fazendo ajuste fino, está consertando um estimador ruim.

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | a árvore sem limite usa 2 594 folhas para 3 000 pontos: $R^2$ de treino 0,9922 e de teste 0,7721 |
| 2 | a floresta **corta $\rho$ pela metade** (0,147 → 0,0735) e paga com $v$ maior (3,28 → 3,80) |
| 2 | o piso $\rho v$ já é 88% da variância com $B=100$ — acrescentar árvores não tem para onde ir |
| 2 | reduzir `max_features` além de $p/3$ continua baixando $\rho$ e **piora** o EQM |
| 3 | de $B=1$ a $B=800$ o risco da floresta só cai e estaciona; o erro de treino para em 0,46 |

**A seguir.** A Aula 06 arruma a casa: `Pipeline`, `ColumnTransformer` e a
disciplina que impede que a padronização, a imputação ou a seleção de variáveis
vejam a dobra de validação.